# LLaMA-2-13B FP16 WikiText-2 Baseline

Evaluate `meta-llama/Llama-2-13b-hf` on the WikiText-2 test split with FP16 weights and activations. No quantization or CPU/disk offload is used.

Requirements: an NVIDIA CUDA GPU with enough VRAM and Hugging Face access to the gated LLaMA-2 repository. An A100 40 GB or larger GPU is recommended.

**Colab L4 support.** The FP16 weights of LLaMA-2-13B (~26 GB) do not fit in an L4 (24 GB). The model is loaded into CPU RAM; embeddings, final norm, `lm_head`, all buffers, and the first decoder layers that fit in `GPU_WEIGHT_BUDGET_GIB` are placed on the GPU, and the remaining decoder layers are streamed from pinned CPU memory right before they run. Every operation, including weight quantization, still runs on the GPU with the same kernels, so results are identical to a fully GPU-resident run; only the runtime grows. Requires at least 40 GiB of system RAM (Colab L4 provides about 53 GB).

In [ ]:
%pip install -q transformers datasets accelerate sentencepiece tqdm

In [ ]:
import json
import os
import platform
import time
from pathlib import Path

import datasets
import psutil
import torch
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-2-13b-hf"
GPU_WEIGHT_BUDGET_GIB = 16  # Colab L4: decoder layers beyond this budget are streamed from CPU RAM.
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = "non_overlapping_2048_drop_remainder"
OUTPUT_PATH = Path("llama2-13b-baseline-s2048.json")

if not torch.cuda.is_available():
    raise RuntimeError("This baseline requires an NVIDIA CUDA GPU.")

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False

MIN_SYSTEM_RAM_GIB = 40
system_ram_gib = psutil.virtual_memory().total / 2**30
if system_ram_gib < MIN_SYSTEM_RAM_GIB:
    raise RuntimeError(
        f"LLaMA-2-13B needs >= {MIN_SYSTEM_RAM_GIB} GiB system RAM; found {system_ram_gib:.1f} GiB."
    )
print(f"System RAM: {system_ram_gib:.1f} GiB")

## Decoder-layer streaming for Colab L4

`DecoderLayerStreamer` keeps the weights that do not fit in `GPU_WEIGHT_BUDGET_GIB` in pinned CPU memory and copies one decoder layer to the GPU right before it runs. `on_compute_device` makes in-place weight quantization run on the GPU even for streamed layers and writes the result back to their CPU weights. Per-module statistics are allocated on `DEVICE` instead of on `weight.device`, because streamed weights live on the CPU between forward passes.

In [ ]:
import functools

DEVICE = torch.device("cuda:0")


class DecoderLayerStreamer:
    """Fit a model whose FP16 weights exceed GPU memory by streaming decoder layers.

    Embeddings, final norm, lm_head, all buffers, and the first decoder layers that fit in
    `gpu_weight_budget_gib` stay on the GPU. The remaining decoder layers keep their weights in
    pinned CPU memory: a forward pre-hook copies a layer's parameters to the GPU right before it
    runs and a forward hook drops the copies afterwards. Every operation still runs on the GPU
    with the same kernels, so outputs are identical to a fully GPU-resident model.
    """

    def __init__(self, model, gpu_weight_budget_gib, device=DEVICE):
        self.device = device
        layers = model.model.layers
        layer_bytes = max(
            sum(p.numel() * p.element_size() for p in layer.parameters()) for layer in layers
        )
        layer_param_ids = {id(p) for layer in layers for p in layer.parameters()}
        other_bytes = sum(
            p.numel() * p.element_size()
            for p in model.parameters() if id(p) not in layer_param_ids
        )
        budget_bytes = gpu_weight_budget_gib * 2**30 - other_bytes
        self.resident_layers = max(0, min(len(layers), int(budget_bytes // layer_bytes)))
        self.streamed_layers = list(layers[self.resident_layers:])
        streamed_ids = {id(p) for layer in self.streamed_layers for p in layer.parameters()}

        with torch.no_grad():
            for module in model.modules():
                for param in module._parameters.values():
                    if param is None:
                        continue
                    if id(param) in streamed_ids:
                        if not param.data.is_pinned():
                            param.data = param.data.to("cpu").pin_memory()
                    elif param.device != device:
                        param.data = param.data.to(device)
                for name, buffer in module._buffers.items():
                    if buffer is not None and buffer.device != device:
                        module._buffers[name] = buffer.to(device)

        self._host = {}
        for layer in self.streamed_layers:
            layer.register_forward_pre_hook(self._fetch)
            layer.register_forward_hook(self._release)
        model.decoder_layer_streamer = self
        print(
            f"Decoder layers: {self.resident_layers} resident on {device}, "
            f"{len(self.streamed_layers)} streamed from pinned CPU memory "
            f"({len(self.streamed_layers) * layer_bytes / 2**30:.2f} GiB per forward pass)"
        )

    def _fetch(self, layer, _args):
        self._release(layer)  # Recover if a previous forward raised (e.g. an auto-batch OOM probe).
        host = []
        for param in layer.parameters():
            if param.device.type == "cpu":
                host.append((param, param.data))
                param.data = param.data.to(self.device, non_blocking=True)
        self._host[id(layer)] = host

    def _release(self, layer, *_):
        for param, data in self._host.pop(id(layer), ()):
            param.data = data

    def check_layout(self, model):
        streamed_ids = {id(p) for layer in self.streamed_layers for p in layer.parameters()}
        for name, param in model.named_parameters():
            if id(param) in streamed_ids:
                assert param.device.type == "cpu" and param.is_pinned(), name
            else:
                assert param.device == self.device, name
        for name, buffer in model.named_buffers():
            assert buffer.device == self.device, name


def on_compute_device(transform, writes_back=True):
    """Run a weight function on DEVICE even when the weight is CPU-resident (streamed).

    With `writes_back=True` an in-place transform's result is copied back into the CPU weight.
    """
    if getattr(transform, "_on_compute_device", False):
        return transform

    @functools.wraps(transform)
    def wrapper(weight, *args, **kwargs):
        if weight.device == DEVICE:
            return transform(weight, *args, **kwargs)
        with torch.no_grad():
            work = weight.detach().to(DEVICE)
            result = transform(work, *args, **kwargs)
            if writes_back:
                weight.copy_(work)
        return result

    wrapper._on_compute_device = True
    return wrapper

## Load the original FP16 model

Before running this cell, accept the model license on Hugging Face and either run `huggingface-cli login` or set the `HF_TOKEN` environment variable. `device_map=0` places the entire model on GPU 0 and prevents automatic CPU/disk offload.

In [ ]:
token = os.getenv("HF_TOKEN")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="cpu",
    attn_implementation="eager",
    token=token,
)
model.eval()
DecoderLayerStreamer(model, GPU_WEIGHT_BUDGET_GIB)

parameter_dtypes = {p.dtype for p in model.parameters() if p.is_floating_point()}
parameter_devices = {p.device.type for p in model.parameters()}
assert parameter_dtypes == {torch.float16}, parameter_dtypes
model.decoder_layer_streamer.check_layout(model)
assert not getattr(model, "is_quantized", False)

print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"Model device: {next(model.parameters()).device}")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = "\n\n".join(dataset["text"])
input_ids = tokenizer(text, return_tensors="pt").input_ids
print(f"WikiText-2 {SPLIT} tokens: {input_ids.numel():,}")

In [ ]:
@torch.inference_mode()
def evaluate_perplexity(model, input_ids, context_length, stride, drop_remainder):
    if stride != context_length:
        raise ValueError("Non-overlapping evaluation requires stride == context_length.")
    if not drop_remainder:
        raise ValueError("Paper-compatible evaluation requires drop_remainder=True.")
    if context_length > model.config.max_position_embeddings:
        raise ValueError("context_length exceeds the model context window.")

    device = next(model.parameters()).device
    sequence_length = input_ids.size(1)
    usable_length = sequence_length // context_length * context_length
    dropped_tokens = sequence_length - usable_length
    if usable_length == 0:
        raise ValueError("Input does not contain a complete context block.")

    total_nll = 0.0
    total_loss_tokens = 0
    total_blocks = usable_length // context_length

    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)
    start_time = time.perf_counter()

    for begin in tqdm(range(0, usable_length, stride), total=total_blocks, desc="Evaluating"):
        end = begin + context_length
        batch = input_ids[:, begin:end].to(device)
        labels = batch.clone()

        loss = model(batch, labels=labels, use_cache=False).loss
        loss_tokens = labels[:, 1:].numel()
        total_nll += loss.float().item() * loss_tokens
        total_loss_tokens += loss_tokens

    torch.cuda.synchronize(device)
    elapsed_seconds = time.perf_counter() - start_time
    mean_nll = total_nll / total_loss_tokens

    return {
        "mean_nll": mean_nll,
        "perplexity": float(torch.exp(torch.tensor(mean_nll))),
        "source_input_tokens": sequence_length,
        "used_input_tokens": usable_length,
        "dropped_input_tokens": dropped_tokens,
        "evaluated_blocks": total_blocks,
        "evaluated_tokens": total_loss_tokens,
        "elapsed_seconds": elapsed_seconds,
        "tokens_per_second": total_loss_tokens / elapsed_seconds,
        "peak_gpu_memory_gib": torch.cuda.max_memory_allocated(device) / 2**30,
    }

In [ ]:
metrics = evaluate_perplexity(model, input_ids, CONTEXT_LENGTH, STRIDE, DROP_REMAINDER)
result = {
    "model": MODEL_ID,
    "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
    "split": SPLIT,
    "dtype": "float16",
    "quantized": False,
    "attention_implementation": "eager",
    "context_length": CONTEXT_LENGTH,
    "stride": STRIDE,
    "evaluation_protocol": EVALUATION_PROTOCOL,
    "drop_remainder": DROP_REMAINDER,
    "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda,
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    **metrics,
}

print(json.dumps(result, indent=2, ensure_ascii=False))
OUTPUT_PATH.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved: {OUTPUT_PATH.resolve()}")

In [ ]:
from google.colab import files
files.download(str(OUTPUT_PATH))